# Lab 01 · Instructor solution

Read the [learner notebook](../01-circuit-dag.ipynb) first. This notebook provides a complete reference and a small discussion example.

Facilitate the learner's reasoning before showing code. Ask them to predict a failure and design a regression test. Passing the shared checks is not evidence by itself of independent understanding.

Use the project's Python 3.12 kernel. This is independent practice material, not a proposed upstream patch.

In [ ]:
import sys

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import Measure, Reset
from qiskit.converters import circuit_to_dag

from qclab.notebook import check_solution

print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Kernel:", sys.executable)
assert qiskit.__version__ == "2.6.0", "Select the project's pinned environment."

In [ ]:
def select_op_nodes(dag, operation_types=None):
    """Return matching operation nodes without modifying the DAG."""
    if operation_types is None:
        return list(dag.op_nodes())
    types = (operation_types,) if isinstance(operation_types, type) else tuple(operation_types)
    if not all(isinstance(operation_type, type) for operation_type in types):
        raise TypeError("Every operation_types entry must be a Python class")
    return [node for node in dag.op_nodes() if isinstance(node.op, types)]

## Why the solution works

1. `None` returns all operation nodes.
2. A class is normalized to one item; an iterable is materialized exactly once.
3. All filter entries are validated before any matching, so neither short-circuiting nor an empty DAG hides invalid input.
4. A single node traversal preserves relative order and prevents duplicate nodes.
5. `isinstance` includes subclasses, including standard singleton gate implementations.

The code uses a Python-level teaching model. Native Qiskit code may use different data structures and Rust bindings.

In [ ]:
result = check_solution(select_op_nodes)
print(result.output)
assert result.passed, "The instructor reference must pass the shared contract."
print("Acceptance status: PASS")

## Example regression discussion

The first `h` operation does not match either filter class. A faulty implementation that repeatedly consumes a generator while inspecting nodes can exhaust the entire generator here. Later measurements and resets then disappear from its results.

In [ ]:
circuit = QuantumCircuit(1, 1)
circuit.h(0)
circuit.measure(0, 0)
circuit.reset(0)
circuit.measure(0, 0)
dag = circuit_to_dag(circuit)

selected = select_op_nodes(dag, (cls for cls in [Measure, Reset]))
assert [node.name for node in selected] == ["measure", "reset", "measure"]
print("Generator regression: PASS")

## Facilitator notes

Record setup time, first confusion, hints used, completion time, and the learner's explanation. Do not count a copied solution as independent completion. See the [facilitator guide](../../docs/facilitator-guide.md).

Prompt the learner to distinguish a class from an instance, an empty filter from `None`, and node order from filter order. Ask what a unitary-equivalence check would miss for a circuit containing measurement and reset; this motivates later correctness lessons without claiming that matrix tests cover all circuit semantics.

The six-lab course is organized around general compiler skills. The author's past PRs are optional case studies, not prerequisites.